In [5]:
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from docx import Document
from docx.shared import Pt


# =============================================================================
# PATHS (en.zip layout)
# =============================================================================
EN_DIR  = Path("../03_Outputs/SEA_Modules/en")
STRUCTURE_PATH = EN_DIR / "module_structure.json"
OUT_DIR = Path("../03_Outputs/SEA_Module_Docx")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("EN_DIR:", EN_DIR.resolve())
print("STRUCTURE_PATH:", STRUCTURE_PATH.resolve())

# =============================================================================
# STRICT TEXT KEYS (only these become visible content)
# =============================================================================
ALLOWED_TEXT_KEYS = {"label", "title", "intro", "text", "body", "description", "value", "prompt"}
EXCLUDE_KEYS = {"cta"}  # never output

# =============================================================================
# SEGMENT / UI FILTERS
# =============================================================================
# Skip entire segments by template_id (no segment names/numbers are ever printed)
SKIP_TEMPLATES = {
    "video",
    "video_full_height",
    "lesson_cover_video",
    "connection_next",
    "module_outro",
    "chapter_outro",
    "lesson_outro",
    "module_intro_cover",
    "chapter_intro_cover",
}

# Drop common navigation/UI strings if they leak in as allowed keys
SKIP_PHRASES = [
    "Next up",
    "Continue learning",
    "Scroll, tab or use your keyboard",
    "What’s next",
    "What's next",
    "This concludes",
    "You have completed",
    "Continue to",
    "Go to next",
    "Click next",
    "Press next",
]

def should_skip_text_value(text: str) -> bool:
    t = (text or "").strip()
    if not t:
        return True
    # Optional: drop URLs
    if t.startswith("http://") or t.startswith("https://"):
        return True
    # Drop common UI phrases
    for p in SKIP_PHRASES:
        if p.lower() in t.lower():
            return True
    return False

# =============================================================================
# CLEANING + RICH TEXT (bold + italic) + AI TAG STRIPPING
# =============================================================================
# We keep:
#   <strong>/<b> -> bold runs
#   <em>/<i> -> italic runs
# We drop:
#   <ai ...>...</ai> (keep inner text only)
#   all other tags
# Also supports markdown-ish _italics_ and **bold** when no HTML is present.
BR_RE = re.compile(r"(?i)<br\s*/?>")
AI_TAG_RE = re.compile(r"(?is)<ai\b[^>]*>(.*?)</ai>")
OTHER_TAG_RE = re.compile(r"<[^>]+>")

# HTML bold/italic spans
HTML_BOLD_RE = re.compile(r"(?is)<(strong|b)>(.*?)</\1>")
HTML_ITALIC_RE = re.compile(r"(?is)<(em|i)>(.*?)</\1>")

# Markdown bold/italic (lightweight)
MD_BOLD_RE = re.compile(r"\*\*(.+?)\*\*")
MD_ITALIC_RE = re.compile(r"(?<!\w)_(.+?)_(?!\w)")

WHITESPACE_RE = re.compile(r"[ \t]+")

def normalize_html_keep_style(text: str) -> str:
    """Convert <br> to newline; strip <ai> tags (keep inner); strip all tags except bold/italic (handled later)."""
    if not text:
        return ""
    text = text.replace("\u00a0", " ")
    text = BR_RE.sub("\n", text)
    # strip ai tags but keep inner
    text = AI_TAG_RE.sub(r"\1", text)
    # normalize whitespace per line (but keep newlines)
    lines = [WHITESPACE_RE.sub(" ", ln).strip() for ln in text.splitlines()]
    return "\n".join([ln for ln in lines if ln])

def strip_all_tags(text: str) -> str:
    return OTHER_TAG_RE.sub("", text or "")

def safe_text(text: str) -> str:
    # after rendering, we want no literal tags remaining
    text = strip_all_tags(text)
    lines = [WHITESPACE_RE.sub(" ", ln).strip() for ln in (text or "").splitlines()]
    return "\n".join([ln for ln in lines if ln])

def parse_inline_runs(s: str) -> List[Tuple[str, Dict[str, bool]]]:
    """
    Return list of (chunk, style_flags) where style_flags can include bold/italic.
    Handles HTML <strong>/<em> and Markdown **bold** and _italic_.
    Priority: HTML tags first; then markdown on remaining plain text.
    """
    # First, expand HTML bold/italic into a token stream.
    # We'll replace HTML tags with placeholders, then post-process.
    chunks: List[Tuple[str, Dict[str, bool]]] = []

    # Helper to process markdown in plain text
    def apply_markdown(text: str, base_style: Dict[str, bool]) -> List[Tuple[str, Dict[str, bool]]]:
        out = []

        # First bold (** **)
        parts = []
        last = 0
        for m in MD_BOLD_RE.finditer(text):
            if m.start() > last:
                parts.append(("plain", text[last:m.start()]))
            parts.append(("bold", m.group(1)))
            last = m.end()
        if last < len(text):
            parts.append(("plain", text[last:]))

        # Then italic (_ _)
        for kind, seg in parts:
            if kind == "bold":
                st = dict(base_style); st["bold"] = True
                out.append((seg, st))
            else:
                last2 = 0
                for m2 in MD_ITALIC_RE.finditer(seg):
                    if m2.start() > last2:
                        out.append((seg[last2:m2.start()], dict(base_style)))
                    st2 = dict(base_style); st2["italic"] = True
                    out.append((m2.group(1), st2))
                    last2 = m2.end()
                if last2 < len(seg):
                    out.append((seg[last2:], dict(base_style)))
        return [(t, st) for (t, st) in out if t]

    def walk_html(text: str, inherited: Dict[str, bool]) -> List[Tuple[str, Dict[str, bool]]]:
        """
        Very lightweight HTML span handling for <strong>/<b> and <em>/<i>.
        We do a loop that finds the earliest of bold/italic tags and recurses on the inner.
        """
        out: List[Tuple[str, Dict[str, bool]]] = []
        # Find earliest tag among bold/italic
        while True:
            mb = HTML_BOLD_RE.search(text)
            mi = HTML_ITALIC_RE.search(text)

            # no more tags
            if not mb and not mi:
                out.extend(apply_markdown(text, inherited))
                break

            # choose the earliest match
            m = None
            tag_type = None
            if mb and mi:
                m, tag_type = (mb, "bold") if mb.start() < mi.start() else (mi, "italic")
            elif mb:
                m, tag_type = mb, "bold"
            else:
                m, tag_type = mi, "italic"

            # before tag
            if m.start() > 0:
                out.extend(apply_markdown(text[:m.start()], inherited))

            inner = m.group(2)
            new_style = dict(inherited)
            new_style[tag_type] = True
            out.extend(walk_html(inner, new_style))

            text = text[m.end():]

        return out

    chunks = walk_html(s, {"bold": False, "italic": False})
    # Clean out any remaining tags in chunks
    cleaned = []
    for t, st in chunks:
        t2 = safe_text(t)
        if t2:
            cleaned.append((t2, st))
    return cleaned

def add_rich_paragraph(doc: Document, text: str, style: Optional[str] = None, bold_all: bool = False, italic_all: bool = False):
    p = doc.add_paragraph(style=style) if style else doc.add_paragraph()
    if bold_all or italic_all:
        r = p.add_run(text)
        r.bold = bold_all
        r.italic = italic_all
        return p

    for chunk, st in parse_inline_runs(text):
        r = p.add_run(chunk)
        if st.get("bold"):
            r.bold = True
        if st.get("italic"):
            r.italic = True
    return p

# =============================================================================
# LIST + SECTION HEADER DETECTION (for readability)
# =============================================================================
BULLET_PREFIX_RE = re.compile(r"^\s*[-•]\s+")
NUMBERED_PREFIX_RE = re.compile(r"^\s*(\d+)\s*[:.)-]\s+")
LETTER_PREFIX_RE = re.compile(r"^\s*([A-Z])\s*[:.)-]\s+")

SECTION_HEADERS = {
    "Key Concepts",
    "Key Takeaways",
    "Key Resources",
    "Learning Objectives",
    "Overview",
}

def render_line(doc: Document, line: str, last_emitted: List[str]):
    """
    Render one cleaned line using heuristics:
      - de-dup immediate repeats
      - bullets if "- " or "• "
      - numbered list if "1: " / "2) " etc.
      - "A: ..." treated as bold section header line
      - known section headers treated as bold line
    """
    line = (line or "").strip()
    if not line:
        return

    # de-dup (immediate)
    normalized = re.sub(r"\s+", " ", safe_text(line)).strip().lower()
    if last_emitted and last_emitted[-1] == normalized:
        return

    # Known section headers -> bold
    if line.strip() in SECTION_HEADERS:
        add_rich_paragraph(doc, safe_text(line), bold_all=True)
        last_emitted.append(normalized)
        return

    # Letter header (A:, B:, etc.) -> bold
    if LETTER_PREFIX_RE.match(line):
        add_rich_paragraph(doc, safe_text(line), bold_all=True)
        last_emitted.append(normalized)
        return

    # Bullet list line
    if BULLET_PREFIX_RE.match(line):
        content = BULLET_PREFIX_RE.sub("", line).strip()
        if content:
            p = doc.add_paragraph(style="List Bullet")
            # rich runs inside bullet
            for chunk, st in parse_inline_runs(content):
                r = p.add_run(chunk)
                r.bold = st.get("bold", False)
                r.italic = st.get("italic", False)
            last_emitted.append(normalized)
        return

    # Numbered list line
    if NUMBERED_PREFIX_RE.match(line):
        content = NUMBERED_PREFIX_RE.sub("", line).strip()
        if content:
            p = doc.add_paragraph(style="List Number")
            for chunk, st in parse_inline_runs(content):
                r = p.add_run(chunk)
                r.bold = st.get("bold", False)
                r.italic = st.get("italic", False)
            last_emitted.append(normalized)
        return

    # Default paragraph
    add_rich_paragraph(doc, safe_text(line))
    last_emitted.append(normalized)

def render_multiline_block(doc: Document, text: str, last_emitted: List[str], force_bold: bool = False):
    for ln in (text or "").split("\n"):
        ln = ln.strip()
        if not ln:
            continue
        if force_bold:
            # still de-dup
            normalized = re.sub(r"\s+", " ", safe_text(ln)).strip().lower()
            if last_emitted and last_emitted[-1] == normalized:
                continue
            add_rich_paragraph(doc, safe_text(ln), bold_all=True)
            last_emitted.append(normalized)
        else:
            render_line(doc, ln, last_emitted)

# =============================================================================
# LOAD JSON + INDEX LESSON JSONs BY TOP-LEVEL id
# =============================================================================
def load_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))

structure = load_json(STRUCTURE_PATH)
assert isinstance(structure, dict) and isinstance(structure.get("modules"), list), "module_structure.json must be {'modules':[...]}"

def build_lesson_index(en_root: Path) -> Dict[str, Path]:
    idx: Dict[str, Path] = {}
    module_dirs = sorted([p for p in en_root.glob("Module_*") if p.is_dir()], key=lambda p: natural_sort_key(p.name))
    all_json = []
    for md in module_dirs:
        all_json.extend([p for p in md.rglob("*.json") if p.is_file() and p.name != "module_structure.json"])
    for p in sorted(all_json, key=lambda x: natural_sort_key(str(x.relative_to(en_root)))):
        try:
            data = json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            continue
        if isinstance(data, dict):
            lid = data.get("id")
            if isinstance(lid, str) and lid.strip():
                idx.setdefault(lid.strip(), p)
    return idx

LESSON_INDEX = build_lesson_index(EN_DIR)
print(f"Indexed {len(LESSON_INDEX)} lesson JSON files under {EN_DIR}")

# =============================================================================
# STRICT TEXT EXTRACTION (only ALLOWED_TEXT_KEYS; exclude CTA)
# =============================================================================
def extract_allowed_text_in_order(obj: Any) -> List[Tuple[str, str]]:
    out: List[Tuple[str, str]] = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in EXCLUDE_KEYS:
                continue
            if k in ALLOWED_TEXT_KEYS and isinstance(v, str):
                txt = normalize_html_keep_style(v).strip()
                if not should_skip_text_value(strip_all_tags(txt)):  # filter based on plain content
                    out.append((k, txt))
                continue
            if isinstance(v, (dict, list)):
                out.extend(extract_allowed_text_in_order(v))
    elif isinstance(obj, list):
        for item in obj:
            out.extend(extract_allowed_text_in_order(item))
    return out

def dedupe_preserve_order(items: List[Tuple[str, str]]) -> List[Tuple[str, str]]:
    seen = set()
    out = []
    for k, t in items:
        key = (k, safe_text(strip_all_tags(t)))
        if key in seen:
            continue
        seen.add(key)
        out.append((k, t))
    return out

# =============================================================================
# DOCX SETUP + RENDER RULES
#   - NO segment headers/numbers
#   - NO CTA
#   - headings bumped up a level: Module=H0, Chapter=H1, Lesson=H2
# =============================================================================
def new_doc() -> Document:
    doc = Document()
    style = doc.styles["Normal"]
    style.font.name = "Calibri"
    style.font.size = Pt(11)
    return doc

def render_items_clean(doc: Document, template_id: str, items: List[Tuple[str, str]], last_emitted: List[str]):
    """
    Clean output:
      - skip templates (video, connection_next, etc.)
      - consecutive 'value' => bullets
      - title/label/prompt => bold lines (not headings)
      - intro/text/body/description => smart paragraphs + list detection
      - remove AI tags; keep bold/italic
      - de-dup immediate repeats across whole doc via last_emitted
    """
    if (template_id or "").strip() in SKIP_TEMPLATES:
        return

    i = 0
    while i < len(items):
        k, t = items[i]

        # consecutive values => bullets (poll options etc.)
        if k == "value":
            vals = []
            j = i
            while j < len(items) and items[j][0] == "value":
                plain = safe_text(strip_all_tags(items[j][1]))
                if not should_skip_text_value(plain):
                    vals.append(items[j][1])
                j += 1
            for v in vals:
                content = safe_text(strip_all_tags(v))
                # de-dup immediate
                normalized = re.sub(r"\s+", " ", content).strip().lower()
                if not content or (last_emitted and last_emitted[-1] == normalized):
                    continue
                p = doc.add_paragraph(style="List Bullet")
                for chunk, st in parse_inline_runs(v):
                    r = p.add_run(chunk)
                    r.bold = st.get("bold", False)
                    r.italic = st.get("italic", False)
                last_emitted.append(normalized)
            i = j
            continue

        # title/label/prompt => bold “section line”, not headings
        if k in {"title", "label", "prompt"}:
            render_multiline_block(doc, safe_text(t), last_emitted, force_bold=True)
        else:
            render_multiline_block(doc, safe_text(t), last_emitted, force_bold=False)

        i += 1

# =============================================================================
# BUILD 1 DOCX PER MODULE
# =============================================================================
missing = []

for mod in structure["modules"]:
    mod_id = str(mod.get("id", "")).strip()
    mod_title_raw = str(mod.get("title", "")).strip()
    mod_title = safe_text(normalize_html_keep_style(mod_title_raw))

    doc = new_doc()
    last_emitted: List[str] = []  # immediate de-dup across whole doc

    # Headings bumped up a level (bigger)
    # Module: H0, Chapter: H1, Lesson: H2
    doc.add_heading(f"Module {mod_id}: {mod_title}", level=0)

    chapters = mod.get("chapters", []) or []
    for ch in chapters:
        ch_id = str(ch.get("id", "")).strip()
        ch_title = safe_text(normalize_html_keep_style(str(ch.get("title", "")).strip()))
        doc.add_heading(f"Chapter {ch_id}: {ch_title}", level=1)

        lessons = ch.get("lessons", []) or []
        for lesson_meta in lessons:
            lesson_id = str(lesson_meta.get("id", "")).strip()
            lesson_title = safe_text(normalize_html_keep_style(str(lesson_meta.get("title", "")).strip()))

            lesson_path = LESSON_INDEX.get(lesson_id)
            if not lesson_path:
                missing.append(lesson_id)
                continue

            doc.add_heading(f"Lesson {lesson_id}: {lesson_title}", level=2)

            try:
                lesson_json = json.loads(lesson_path.read_text(encoding="utf-8"))
            except Exception:
                missing.append(lesson_id)
                continue

            segments = lesson_json.get("segments", [])
            if not isinstance(segments, list):
                segments = []

            for seg in segments:
                if not isinstance(seg, dict):
                    continue
                template_id = str(seg.get("template_id", "")).strip()
                extracted = dedupe_preserve_order(extract_allowed_text_in_order(seg))
                if extracted:
                    render_items_clean(doc, template_id, extracted, last_emitted)

            doc.add_paragraph("")  # lesson spacer

    out_name = f"M{mod_id} - {safe_filename(mod_title)}.docx" if mod_id else f"{safe_filename(mod_title)}.docx"
    out_path = OUT_DIR / out_name
    doc.save(out_path)
    print("Wrote:", out_path)

missing_unique = sorted(set(missing), key=natural_sort_key)
print("\nMissing lesson JSON ids (skipped):", len(missing_unique))
print("First 30 missing (if any):", missing_unique[:30])

EN_DIR: /Users/ben/Documents/UNDP/SEH/Sustainable Energy Academy/Academy_Pipeline/dsc-energy-academy-pipeline/03_Outputs/SEA_Modules/en
STRUCTURE_PATH: /Users/ben/Documents/UNDP/SEH/Sustainable Energy Academy/Academy_Pipeline/dsc-energy-academy-pipeline/03_Outputs/SEA_Modules/en/module_structure.json
Indexed 211 lesson JSON files under ../03_Outputs/SEA_Modules/en
Wrote: ../03_Outputs/SEA_Module_Docx/M1 - Intro to Sustainable Energy for Development.docx
Wrote: ../03_Outputs/SEA_Module_Docx/M2 - Energy Access and Inclusive Energy Services.docx
Wrote: ../03_Outputs/SEA_Module_Docx/M3 - Just Energy Transition.docx
Wrote: ../03_Outputs/SEA_Module_Docx/M4 - Sustainable Energy Finance.docx
Wrote: ../03_Outputs/SEA_Module_Docx/M5 - Sustainable Energy Governance.docx
Wrote: ../03_Outputs/SEA_Module_Docx/M6 - The Climate Change-Energy Nexus.docx
Wrote: ../03_Outputs/SEA_Module_Docx/M7 - Data in Sustainable Energy.docx
Wrote: ../03_Outputs/SEA_Module_Docx/M8 - The Future of Energy Innovation for

In [9]:
import json
import re
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import urlparse

import requests
from PIL import Image, UnidentifiedImageError  # pillow
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH


# =============================================================================
# PATHS (en.zip layout)
# =============================================================================
EN_DIR  = Path("../03_Outputs/SEA_Modules/en")
STRUCTURE_PATH = EN_DIR / "module_structure.json"
OUT_DIR = Path("../03_Outputs/SEA_Module_Docx")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Download cache for Azure images (as requested)
IMAGE_TEMP_DIR = Path("../03_Outputs/image_temp")
IMAGE_TEMP_DIR.mkdir(parents=True, exist_ok=True)

print("EN_DIR:", EN_DIR.resolve())
print("STRUCTURE_PATH:", STRUCTURE_PATH.resolve())
print("OUT_DIR:", OUT_DIR.resolve())
print("IMAGE_TEMP_DIR:", IMAGE_TEMP_DIR.resolve())

# =============================================================================
# DOCX IMAGE SETTINGS
# =============================================================================
IMAGE_WIDTH = Inches(6.3)

# python-docx supported formats
DOCX_OK_EXTS = {".png", ".jpg", ".jpeg", ".gif", ".bmp", ".tif", ".tiff"}

# =============================================================================
# STRICT TEXT KEYS (only these become visible text content)
# =============================================================================
ALLOWED_TEXT_KEYS = {"label", "title", "intro", "text", "body", "description", "value", "prompt"}
EXCLUDE_KEYS = {"cta"}  # never output

# =============================================================================
# SEGMENT / UI FILTERS
# =============================================================================
SKIP_TEMPLATES = {
    "video",
    "video_full_height",
    "lesson_cover_video",
    "connection_next",
    "module_outro",
    "chapter_outro",
    "lesson_outro",
    "module_intro_cover",
    "chapter_intro_cover",
}

# ✅ ONLY insert images for these templates
IMAGE_TEMPLATES = {"infographic", "chart"}

SKIP_PHRASES = [
    "Next up",
    "Continue learning",
    "Scroll, tab or use your keyboard",
    "What’s next",
    "What's next",
    "This concludes",
    "You have completed",
    "Continue to",
    "Go to next",
    "Click next",
    "Press next",
]

def should_skip_text_value(text: str) -> bool:
    t = (text or "").strip()
    if not t:
        return True
    if t.startswith("http://") or t.startswith("https://"):
        return True
    for p in SKIP_PHRASES:
        if p.lower() in t.lower():
            return True
    return False

# =============================================================================
# CLEANING + RICH TEXT (bold + italic) + AI TAG STRIPPING
# =============================================================================
BR_RE = re.compile(r"(?i)<br\s*/?>")
AI_TAG_RE = re.compile(r"(?is)<ai\b[^>]*>(.*?)</ai>")
OTHER_TAG_RE = re.compile(r"<[^>]+>")

HTML_BOLD_RE = re.compile(r"(?is)<(strong|b)>(.*?)</\1>")
HTML_ITALIC_RE = re.compile(r"(?is)<(em|i)>(.*?)</\1>")

MD_BOLD_RE = re.compile(r"\*\*(.+?)\*\*")
MD_ITALIC_RE = re.compile(r"(?<!\w)_(.+?)_(?!\w)")

WHITESPACE_RE = re.compile(r"[ \t]+")

def normalize_html_keep_style(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\u00a0", " ")
    text = BR_RE.sub("\n", text)
    text = AI_TAG_RE.sub(r"\1", text)
    lines = [WHITESPACE_RE.sub(" ", ln).strip() for ln in text.splitlines()]
    return "\n".join([ln for ln in lines if ln])

def strip_all_tags(text: str) -> str:
    return OTHER_TAG_RE.sub("", text or "")

def safe_text(text: str) -> str:
    text = strip_all_tags(text)
    lines = [WHITESPACE_RE.sub(" ", ln).strip() for ln in (text or "").splitlines()]
    return "\n".join([ln for ln in lines if ln])

def parse_inline_runs(s: str) -> List[Tuple[str, Dict[str, bool]]]:
    def apply_markdown(text: str, base_style: Dict[str, bool]) -> List[Tuple[str, Dict[str, bool]]]:
        out = []
        parts = []
        last = 0
        for m in MD_BOLD_RE.finditer(text):
            if m.start() > last:
                parts.append(("plain", text[last:m.start()]))
            parts.append(("bold", m.group(1)))
            last = m.end()
        if last < len(text):
            parts.append(("plain", text[last:]))

        for kind, seg in parts:
            if kind == "bold":
                st = dict(base_style); st["bold"] = True
                out.append((seg, st))
            else:
                last2 = 0
                for m2 in MD_ITALIC_RE.finditer(seg):
                    if m2.start() > last2:
                        out.append((seg[last2:m2.start()], dict(base_style)))
                    st2 = dict(base_style); st2["italic"] = True
                    out.append((m2.group(1), st2))
                    last2 = m2.end()
                if last2 < len(seg):
                    out.append((seg[last2:], dict(base_style)))
        return [(t, st) for (t, st) in out if t]

    def walk_html(text: str, inherited: Dict[str, bool]) -> List[Tuple[str, Dict[str, bool]]]:
        out: List[Tuple[str, Dict[str, bool]]] = []
        while True:
            mb = HTML_BOLD_RE.search(text)
            mi = HTML_ITALIC_RE.search(text)
            if not mb and not mi:
                out.extend(apply_markdown(text, inherited))
                break

            if mb and mi:
                m, tag_type = (mb, "bold") if mb.start() < mi.start() else (mi, "italic")
            elif mb:
                m, tag_type = mb, "bold"
            else:
                m, tag_type = mi, "italic"

            if m.start() > 0:
                out.extend(apply_markdown(text[:m.start()], inherited))

            inner = m.group(2)
            new_style = dict(inherited)
            new_style[tag_type] = True
            out.extend(walk_html(inner, new_style))

            text = text[m.end():]
        return out

    chunks = walk_html(s or "", {"bold": False, "italic": False})
    cleaned = []
    for t, st in chunks:
        t2 = safe_text(t)
        if t2:
            cleaned.append((t2, st))
    return cleaned

def add_rich_paragraph(doc: Document, text: str, style: Optional[str] = None, bold_all: bool = False):
    p = doc.add_paragraph(style=style) if style else doc.add_paragraph()
    if bold_all:
        r = p.add_run(text)
        r.bold = True
        return p
    for chunk, st in parse_inline_runs(text):
        r = p.add_run(chunk)
        r.bold = bool(st.get("bold"))
        r.italic = bool(st.get("italic"))
    return p

# =============================================================================
# LIST + SECTION HEADER DETECTION
# =============================================================================
BULLET_PREFIX_RE = re.compile(r"^\s*[-•]\s+")
NUMBERED_PREFIX_RE = re.compile(r"^\s*(\d+)\s*[:.)-]\s+")
LETTER_PREFIX_RE = re.compile(r"^\s*([A-Z])\s*[:.)-]\s+")

SECTION_HEADERS = {
    "Key Concepts",
    "Key Takeaways",
    "Key Resources",
    "Learning Objectives",
    "Overview",
}

def render_line(doc: Document, line: str, last_emitted: List[str]):
    line = (line or "").strip()
    if not line:
        return

    normalized = re.sub(r"\s+", " ", safe_text(line)).strip().lower()
    if last_emitted and last_emitted[-1] == normalized:
        return

    if line.strip() in SECTION_HEADERS or LETTER_PREFIX_RE.match(line):
        add_rich_paragraph(doc, safe_text(line), bold_all=True)
        last_emitted.append(normalized)
        return

    if BULLET_PREFIX_RE.match(line):
        content = BULLET_PREFIX_RE.sub("", line).strip()
        if content:
            p = doc.add_paragraph(style="List Bullet")
            for chunk, st in parse_inline_runs(content):
                r = p.add_run(chunk)
                r.bold = bool(st.get("bold"))
                r.italic = bool(st.get("italic"))
            last_emitted.append(normalized)
        return

    if NUMBERED_PREFIX_RE.match(line):
        content = NUMBERED_PREFIX_RE.sub("", line).strip()
        if content:
            p = doc.add_paragraph(style="List Number")
            for chunk, st in parse_inline_runs(content):
                r = p.add_run(chunk)
                r.bold = bool(st.get("bold"))
                r.italic = bool(st.get("italic"))
            last_emitted.append(normalized)
        return

    add_rich_paragraph(doc, safe_text(line))
    last_emitted.append(normalized)

def render_multiline_block(doc: Document, text: str, last_emitted: List[str], force_bold: bool = False):
    for ln in (text or "").split("\n"):
        ln = ln.strip()
        if not ln:
            continue
        if force_bold:
            norm = re.sub(r"\s+", " ", safe_text(ln)).strip().lower()
            if last_emitted and last_emitted[-1] == norm:
                continue
            add_rich_paragraph(doc, safe_text(ln), bold_all=True)
            last_emitted.append(norm)
        else:
            render_line(doc, ln, last_emitted)

# =============================================================================
# JSON LOAD + INDEX LESSON JSONs
# =============================================================================
def natural_sort_key(s: str):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(s))]

def safe_filename(s: str, maxlen: int = 90) -> str:
    s = re.sub(r"[^A-Za-z0-9 _-]+", "", (s or "")).strip()
    s = re.sub(r"\s+", " ", s)
    return (s[:maxlen].rstrip() if s else "Module").strip()

def load_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))

structure = load_json(STRUCTURE_PATH)
assert isinstance(structure, dict) and isinstance(structure.get("modules"), list), "module_structure.json must be {'modules':[...]}"

def build_lesson_index(en_root: Path) -> Dict[str, Path]:
    idx: Dict[str, Path] = {}
    module_dirs = sorted([p for p in en_root.glob("Module_*") if p.is_dir()], key=lambda p: natural_sort_key(p.name))
    all_json = []
    for md in module_dirs:
        all_json.extend([p for p in md.rglob("*.json") if p.is_file() and p.name != "module_structure.json"])
    for p in sorted(all_json, key=lambda x: natural_sort_key(str(x.relative_to(en_root)))):
        try:
            data = json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            continue
        if isinstance(data, dict):
            lid = data.get("id")
            if isinstance(lid, str) and lid.strip():
                idx.setdefault(lid.strip(), p)
    return idx

LESSON_INDEX = build_lesson_index(EN_DIR)
print(f"Indexed {len(LESSON_INDEX)} lesson JSON files under ../03_Outputs/SEA_Modules/en")

# =============================================================================
# STRICT TEXT EXTRACTION
# =============================================================================
def extract_allowed_text_in_order(obj: Any) -> List[Tuple[str, str]]:
    out: List[Tuple[str, str]] = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in EXCLUDE_KEYS:
                continue
            if k in ALLOWED_TEXT_KEYS and isinstance(v, str):
                txt = normalize_html_keep_style(v).strip()
                plain = safe_text(txt)
                if not should_skip_text_value(plain):
                    out.append((k, txt))
                continue
            if isinstance(v, (dict, list)):
                out.extend(extract_allowed_text_in_order(v))
    elif isinstance(obj, list):
        for item in obj:
            out.extend(extract_allowed_text_in_order(item))
    return out

def dedupe_preserve_order(items: List[Tuple[str, str]]) -> List[Tuple[str, str]]:
    seen = set()
    out = []
    for k, t in items:
        key = (k, safe_text(t))
        if key in seen:
            continue
        seen.add(key)
        out.append((k, t))
    return out

# =============================================================================
# AZURE IMAGE URL EXTRACTION + DOWNLOAD + CONVERT (WEBP -> PNG)
# =============================================================================
URL_RE = re.compile(r"https?://[^\s)>\"]+")
IMAGE_LIKE = (".png", ".jpg", ".jpeg", ".webp", ".gif", ".bmp", ".tif", ".tiff")

def looks_like_image_url(u: str) -> bool:
    try:
        parsed = urlparse(u)
        if parsed.scheme not in {"http", "https"}:
            return False
        path = parsed.path.lower()
        return any(ext in path for ext in IMAGE_LIKE)
    except Exception:
        return False

def extract_image_urls_from_segment(seg: Dict[str, Any]) -> List[str]:
    urls: List[str] = []
    def walk(o: Any):
        if isinstance(o, dict):
            for v in o.values():
                walk(v)
        elif isinstance(o, list):
            for it in o:
                walk(it)
        elif isinstance(o, str):
            for m in URL_RE.finditer(o):
                u = m.group(0).strip().rstrip(".,;")
                if looks_like_image_url(u):
                    urls.append(u)
    walk(seg)
    seen, out = set(), []
    for u in urls:
        if u in seen:
            continue
        seen.add(u)
        out.append(u)
    return out

def cache_path_for_url(url: str) -> Path:
    h = hashlib.sha256(url.encode("utf-8")).hexdigest()[:24]
    ext = Path(urlparse(url).path).suffix.lower() or ".bin"
    return IMAGE_TEMP_DIR / f"{h}{ext}"

def download_url_to_cache(url: str, timeout: int = 60) -> Optional[Path]:
    out_path = cache_path_for_url(url)
    if out_path.exists() and out_path.stat().st_size > 0:
        return out_path

    try:
        r = requests.get(url, stream=True, timeout=timeout)
        r.raise_for_status()

        if out_path.suffix in {".bin", ""}:
            ct = (r.headers.get("Content-Type") or "").lower()
            if "png" in ct: out_path = out_path.with_suffix(".png")
            elif "jpeg" in ct or "jpg" in ct: out_path = out_path.with_suffix(".jpg")
            elif "webp" in ct: out_path = out_path.with_suffix(".webp")
            elif "gif" in ct: out_path = out_path.with_suffix(".gif")
            else: out_path = out_path.with_suffix(".bin")

        tmp = out_path.with_suffix(out_path.suffix + ".part")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 256):
                if chunk:
                    f.write(chunk)
        tmp.replace(out_path)
        return out_path
    except Exception:
        return None

def ensure_docx_compatible_image(path: Path) -> Optional[Path]:
    if not path or not path.exists() or path.stat().st_size == 0:
        return None

    # Reject HTML error pages quickly
    try:
        with open(path, "rb") as f:
            head = f.read(64)
        if head.lstrip().startswith(b"<"):
            return None
    except Exception:
        pass

    try:
        img = Image.open(path)
        img.load()
    except UnidentifiedImageError:
        return None
    except Exception:
        return None

    if path.suffix.lower() in DOCX_OK_EXTS:
        return path

    out_png = path.with_suffix(".png")
    try:
        if img.mode not in ("RGB", "RGBA"):
            img = img.convert("RGB")
        img.save(out_png, format="PNG")
        return out_png
    except Exception:
        return None

def insert_image(doc: Document, path: Path, caption: Optional[str] = None):
    if not path or not path.exists():
        return
    p = doc.add_paragraph()
    run = p.add_run()
    run.add_picture(str(path), width=IMAGE_WIDTH)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER

    if caption:
        cap = safe_text(caption)
        if cap:
            pc = doc.add_paragraph()
            rc = pc.add_run(cap)
            rc.italic = True
            pc.alignment = WD_ALIGN_PARAGRAPH.CENTER

# =============================================================================
# DOCX SETUP + RENDER RULES
# =============================================================================
def new_doc() -> Document:
    doc = Document()
    style = doc.styles["Normal"]
    style.font.name = "Calibri"
    style.font.size = Pt(11)
    return doc

def render_items_clean(doc: Document, items: List[Tuple[str, str]], last_emitted: List[str]):
    i = 0
    while i < len(items):
        k, t = items[i]

        if k == "value":
            vals = []
            j = i
            while j < len(items) and items[j][0] == "value":
                plain = safe_text(items[j][1])
                if not should_skip_text_value(plain):
                    vals.append(items[j][1])
                j += 1

            for v in vals:
                content_plain = safe_text(v)
                normalized = re.sub(r"\s+", " ", content_plain).strip().lower()
                if not content_plain or (last_emitted and last_emitted[-1] == normalized):
                    continue
                p = doc.add_paragraph(style="List Bullet")
                for chunk, st in parse_inline_runs(v):
                    r = p.add_run(chunk)
                    r.bold = bool(st.get("bold"))
                    r.italic = bool(st.get("italic"))
                last_emitted.append(normalized)

            i = j
            continue

        if k in {"title", "label", "prompt"}:
            render_multiline_block(doc, safe_text(t), last_emitted, force_bold=True)
        else:
            render_multiline_block(doc, safe_text(t), last_emitted, force_bold=False)

        i += 1

# =============================================================================
# BUILD 1 DOCX PER MODULE (insert images ONLY for chart/infographic)
# =============================================================================
missing = []

for mod in structure["modules"]:
    mod_id = str(mod.get("id", "")).strip()
    mod_title = safe_text(normalize_html_keep_style(str(mod.get("title", "")).strip()))

    doc = new_doc()
    last_emitted: List[str] = []

    doc.add_heading(f"Module {mod_id}: {mod_title}", level=0)

    for ch in (mod.get("chapters", []) or []):
        ch_id = str(ch.get("id", "")).strip()
        ch_title = safe_text(normalize_html_keep_style(str(ch.get("title", "")).strip()))
        doc.add_heading(f"Chapter {ch_id}: {ch_title}", level=1)

        for lesson_meta in (ch.get("lessons", []) or []):
            lesson_id = str(lesson_meta.get("id", "")).strip()
            lesson_title = safe_text(normalize_html_keep_style(str(lesson_meta.get("title", "")).strip()))

            lesson_path = LESSON_INDEX.get(lesson_id)
            if not lesson_path:
                missing.append(lesson_id)
                continue

            doc.add_heading(f"Lesson {lesson_id}: {lesson_title}", level=2)

            try:
                lesson_json = json.loads(lesson_path.read_text(encoding="utf-8"))
            except Exception:
                missing.append(lesson_id)
                continue

            segments = lesson_json.get("segments", [])
            if not isinstance(segments, list):
                segments = []

            for seg in segments:
                if not isinstance(seg, dict):
                    continue

                template_id = str(seg.get("template_id", "")).strip()

                if template_id in SKIP_TEMPLATES:
                    continue

                # ✅ ONLY charts/infographics
                if template_id in IMAGE_TEMPLATES:
                    urls = extract_image_urls_from_segment(seg)
                    if urls:
                        caption = None
                        content = seg.get("content", {})
                        if isinstance(content, dict):
                            cap = content.get("title") or content.get("label") or content.get("description")
                            if isinstance(cap, str):
                                caption = normalize_html_keep_style(cap)

                        for url in urls:
                            downloaded = download_url_to_cache(url)
                            if not downloaded:
                                continue
                            compatible = ensure_docx_compatible_image(downloaded)
                            if not compatible:
                                continue
                            try:
                                insert_image(doc, compatible, caption=caption)
                            except Exception:
                                continue

                extracted = dedupe_preserve_order(extract_allowed_text_in_order(seg))
                if extracted:
                    render_items_clean(doc, extracted, last_emitted)

            doc.add_paragraph("")

    out_name = f"M{mod_id} - {safe_filename(mod_title)}.docx" if mod_id else f"{safe_filename(mod_title)}.docx"
    out_path = OUT_DIR / out_name
    doc.save(out_path)
    print("Wrote:", out_path)

missing_unique = sorted(set(missing), key=natural_sort_key)
print("\nMissing lesson JSON ids (skipped):", len(missing_unique))
print("First 30 missing (if any):", missing_unique[:30])

print("\nDone. Images inserted ONLY for templates:", IMAGE_TEMPLATES)
print("Cache dir:", IMAGE_TEMP_DIR.resolve())

EN_DIR: /Users/ben/Documents/UNDP/SEH/Sustainable Energy Academy/Academy_Pipeline/dsc-energy-academy-pipeline/03_Outputs/SEA_Modules/en
STRUCTURE_PATH: /Users/ben/Documents/UNDP/SEH/Sustainable Energy Academy/Academy_Pipeline/dsc-energy-academy-pipeline/03_Outputs/SEA_Modules/en/module_structure.json
OUT_DIR: /Users/ben/Documents/UNDP/SEH/Sustainable Energy Academy/Academy_Pipeline/dsc-energy-academy-pipeline/03_Outputs/SEA_Module_Docx
IMAGE_TEMP_DIR: /Users/ben/Documents/UNDP/SEH/Sustainable Energy Academy/Academy_Pipeline/dsc-energy-academy-pipeline/03_Outputs/image_temp
Indexed 211 lesson JSON files under ../03_Outputs/SEA_Modules/en
Wrote: ../03_Outputs/SEA_Module_Docx/M1 - Intro to Sustainable Energy for Development.docx
Wrote: ../03_Outputs/SEA_Module_Docx/M2 - Energy Access and Inclusive Energy Services.docx
Wrote: ../03_Outputs/SEA_Module_Docx/M3 - Just Energy Transition.docx
Wrote: ../03_Outputs/SEA_Module_Docx/M4 - Sustainable Energy Finance.docx
Wrote: ../03_Outputs/SEA_Modu